In [ ]:
import numpy as np
import pandas as pd

def softmax(x: np.ndarray, axis=-1):
    """
    softmax turns arbitrary real numbers into a probability distribution. 

    # we would need this to convert aspect scores phi into proper preference vector probabilities.

    Example:
        x = [2, 1, 0]
        softmax(x) -> [0.66, 0.24, 0.09] (roughly)

    Why subtract max?
        Prevents exp(large number) overflow.
    """
    x = np.asarray(x)
    x = x - np.max(x, axis=axis, keepdims=True)  # numerical stability
    ex = np.exp(x)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + 1e-12)


def ensure_length_col(df: pd.DataFrame, text_col="review_text", length_col="len"):
    """
    Adds a column df[length_col] = length of each sentence string.

    Why?
      - We enforce length budget L in extraction
      - We optionally penalize too-long sentences

    If already present, does nothing.
    """
    if length_col not in df.columns:
        df = df.copy()
        df[length_col] = df[text_col].fillna("").astype(str).str.len().astype(int)
    return df

# DONE in extractors/common.py : get_phi_matrix
def get_phi_matrix(df: pd.DataFrame, phi_cols):
    """
    Extract aspect-score columns from df and return a numpy matrix Phi.

    Phi shape: (N, K)
      - N = number of rows (sentences/reviews)
      - K = number of aspects

    We also row-normalize so each row sums to 1 (simplex).
    """
    Phi = df[phi_cols].to_numpy(dtype=np.float32)
    # (optional) row-normalize if you want exact simplex rows # phi_i/sum(phi_i)
    row_sums = Phi.sum(axis=1, keepdims=True)
    Phi = Phi / (row_sums + 1e-12)
    return Phi


In [ ]:
# Section 3.3 : importance weighted aspect profile with feedback weighting
# DONE in preference.py
def aspect_profile(phi_rows: np.ndarray, idx: np.ndarray, alpha: np.ndarray | None = None):
    """
    Compute aspect profile z_t for a selected set S_t (importance weighted).

    Inputs:
      phi_rows: (N, K) full aspect matrix for all rows in your dataset
      idx: list/array of indices that represent the selected set S_t
      alpha: optional weights per selected sentence (importance weighting)

    Output:
      z_t: (K,) vector representing aspect distribution of that selection
    """
    if len(idx) == 0:
        raise ValueError("Empty S_t.")
    if alpha is None:
        return phi_rows[idx].mean(axis=0)
    alpha = np.asarray(alpha, dtype=np.float32)
    alpha = alpha / (alpha.sum() + 1e-12) # sum of alpha_i should be 1 for proper weighting
    return (phi_rows[idx] * alpha[:, None]).sum(axis=0) # weighted average of aspect profiles for selected sentences

# DONE in preference.py
def estimate_user_pref_from_history(history, phi_rows, beta=5.0, eps=1e-8, prior=None):
    """
    Estimate the user preference vector w_u from interaction history using an empirical Boltzmann estimation.

    history: list of events, each event dict contains:
      {
        "S_idx": np.array([...])   # indices of selected sentences (S_t)
        "f": float                 # feedback in [0,1]
        "alpha": optional weights over S_t
      }

    phi_rows: (N, K) full aspect matrix
    beta: preference sharpness for softmax
    prior: optional prior distribution over aspects (K,)

    Output:
      w_u: (K,) user preference vector in simplex
    """
    K = phi_rows.shape[1]

    # if no history: return prior or uniform
    if (history is None) or (len(history) == 0):
        if prior is None:
            return np.ones(K, dtype=np.float32) / K
        prior = np.asarray(prior, dtype=np.float32)
        return prior / (prior.sum() + 1e-12)

    # numerator and denominator for ztilde_u
    num = np.zeros(K, dtype=np.float32)
    den = 0.0

    # build: ztilde_u = sum_t f_t z_t / sum_t f_t
    # for each event in history, compute aspect profile z_t and accumulate weighted by feedback f_t
    for ev in history:
        f = float(ev["f"])
        S_idx = np.asarray(ev["S_idx"], dtype=int)
        alpha = ev.get("alpha", None)

        z_t = aspect_profile(phi_rows, S_idx, alpha=alpha)  # (K,) aspect profile for this event
        num += f * z_t # accumulate weighted aspect profile
        den += f # accumulate total feedback for normalization

    z_tilde = num / (den + eps)  # (K,) z_tilde_u =(sum_t f_t z_t) / (sum_t f_t + eps) for stability ... this is the feedback-weighted average aspect profile across history

    # Boltzmann / softmax mapping to simplex # section 3.3.1
    w = softmax(beta * z_tilde, axis=0)  

    # optional blending with prior
    if prior is not None:
        prior = np.asarray(prior, dtype=np.float32)
        prior = prior / (prior.sum() + 1e-12)
        w = 0.9 * w + 0.1 * prior
        w = w / (w.sum() + 1e-12)

    return w # output w_u of shape K summing to 1 (simplex)

# here aspects that appear in sets with higher feedback f_t will have higher z_tilde values, and the softmax with beta will amplify the differences to produce a sharper preference vector w_u.

In [ ]:
# DONE in preference.py
# section 3.3 uniform averaging without feedback weighting
def estimate_user_pref_from_own_text(df_sent, user_id, phi_cols, beta=5.0):
    """
    Bootstraps w_u directly from the user's own sentences/reviews  if present 
    """
    sub = df_sent[df_sent["user_id"] == user_id]
    if len(sub) == 0:
        K = len(phi_cols)
        return np.ones(K, dtype=np.float32) / K
    Phi = get_phi_matrix(sub, phi_cols)      # (m,K)
    ztilde = Phi.mean(axis=0)                # average aspect mass
    return softmax(beta * ztilde, axis=0)

In [4]:
# this is section 3.1
def compute_sentence_utilities(df_candidates, phi_cols, w_u, lam=0.0, len_col="len"):
    """
    For a product p, we have candidate sentences.

    Compute for each sentence i:
      base_i = w_u^T phi_i   (how much it matches the user's aspects)

    Optionally penalize long sentences:
      base_i -= lam * len_i / mean_len

    This is the relevance score U_i for each candidate sentence.
    """
    Phi = get_phi_matrix(df_candidates, phi_cols)         # (n_p, K)
    w_u = np.asarray(w_u, dtype=np.float32)               # (K,)

    # aspect relevance score per sentence
    U = Phi @ w_u                                         # (n_p,)

    # optional length penalty
    if lam > 0:
        lens = df_candidates[len_col].to_numpy(dtype=np.float32)
        mean_len = float(np.mean(lens) + 1e-12)
        U = U - lam * (lens / mean_len)

    return U, Phi


def softmax_sentence_policy(U, tau_ext=1.0): # never used currently this section 3.1.2 equation (7)
    logits = tau_ext * np.asarray(U, dtype=np.float32)
    return softmax(logits, axis=0)


In [ ]:
# DONE in extractors/gumbel.py
def sample_gumbel(shape, rng=None):
    """
    Generate Gumbel(0,1) noise. Section 3.1.2  describes g_{t',j}

    If U ~ Uniform(0,1), then:
      G = -log(-log(U)) is ensures Gumbel distribution
    """
    rng = np.random.default_rng() if rng is None else rng
    U = rng.uniform(low=1e-12, high=1.0 - 1e-12, size=shape)
    return -np.log(-np.log(U))

# DONE in extractors/gumbel.py
def gumbel_priority_order(U, tau_ext=1.0, seed=0):
    """
    Create a random ordering of sentences where higher-utility sentences
    tend to appear earlier.

    # section 3.1.2 describes this process of creating a random priority ordering using gumbel-perturbed utilities.
    Steps:
      1) sample g_i ~ Gumbel
      2) xi_i = tau_ext * U_i + g_i
      3) sort by xi in descending order

    Output:
      perm: indices sorted high-to-low
      xi: perturbed scores
    """
    rng = np.random.default_rng(seed)
    g = sample_gumbel(len(U), rng=rng)
    xi = tau_ext * np.asarray(U, dtype=np.float32) + g
    perm = np.argsort(-xi)  # descending
    return perm, xi

# DONE in extractors/gumbel.py
def gumbel_priority_greedy_select(df_candidates, U, k=8, L=800, len_col="len", tau_ext=1.0, seed=0):
    """
    Implements algorithm 2 : section 3.1.2

    - create random priority ordering using gumbel-perturbed utilities
    - scan in that order
    - add sentence if it doesn't violate constraints:
        (1) |S| <= k
        (2) total length <= L

    Output:
      selected_local_idx: indices (within df_candidates) that were selected
      perm: the full ordering
      xi: reminder scores
      total_len: length used
    """
    perm, xi = gumbel_priority_order(U, tau_ext=tau_ext, seed=seed)

    selected = []
    total_len = 0
    lens = df_candidates[len_col].to_numpy(dtype=int)

    for j in perm:
        if len(selected) >= k:
            break
        if total_len + int(lens[j]) <= L:
            selected.append(j)
            total_len += int(lens[j])

    return np.array(selected, dtype=int), perm, xi, total_len


In [6]:
def extract_for_user_product(
    df_sent: pd.DataFrame,
    user_id: str,
    product_id: str,
    phi_cols,
    w_u: np.ndarray,
    k=8,
    L=800,
    lam=0.0,
    tau_ext=1.0,
    seed=0,
    text_col="review_text",
    len_col="len",
):
    """
    Main function to call for algorithm 2 implementation : Section 3.1.2

    Inputs:
      df_sent: full sentence table (contains parent_asin, sentence text, phi_k columns)
      user_id: currently used only for logging/extension (not required here)
      product_id: product we want to summarize
      w_u: user's preference vector
      k: max number of sentences to select
      L: max total length
      lam: length penalty weight
      tau_ext: softmax sharpness in Gumbel ranking
      seed: randomness control

    Outputs:
      selected_df: the extracted set of sentences
      z_t: aspect profile of extracted set (average phi)
    """
    # ensure we have lengths
    df_sent = ensure_length_col(df_sent, text_col=text_col, length_col=len_col)

    # candidate sentences for this product
    cand = df_sent[df_sent["parent_asin"] == product_id].copy()
    if len(cand) == 0:
        return None

    # compute utilities U_i and also Phi for these candidates
    U, Phi = compute_sentence_utilities(cand, phi_cols, w_u, lam=lam, len_col=len_col)

    # run constrained gumbel-greedy extraction
    sel_local_idx, perm, xi, total_len = gumbel_priority_greedy_select(
        cand, U, k=k, L=L, len_col=len_col, tau_ext=tau_ext, seed=seed
    )

    # selected rows
    selected_df = cand.iloc[sel_local_idx].copy()
    selected_df["utility"] = U[sel_local_idx]
    selected_df["gumbel_score"] = xi[sel_local_idx]

    # aspect profile of selected set (uniform avg)    
    z_t = Phi[sel_local_idx].mean(axis=0)

    return {
        "selected_df": selected_df.sort_values("gumbel_score", ascending=False),
        "z_t": z_t,
        "total_len": total_len,
        "n_candidates": len(cand),
    }

In [7]:
data = pd.read_csv("../data/train_with_aspect_scores.csv")
data.head()

,user_id,parent_asin,rating,timestamp,history,review_title,review_text,helpful_vote,verified_purchase,main_category,...,asp_12,asp_13,asp_14,asp_15,asp_16,asp_17,asp_18,asp_19,top_aspect,top_aspect_score
0,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B081TJ8YS3,4.0,1588615855,NaN,Works great but smells a little weird.,"This product does what I need it to do, I just...",1,True,All Beauty,...,0.000672,0.000268,0.017212,0.004381,0.000119,0.739653,0.000900,0.000672,asp_17,0.739653
1,AGKHLEW2SOWHNMFQIJGBECAF7INQ,B00YQ6X8EO,5.0,1588687728,B081TJ8YS3,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,0,True,All Beauty,...,0.000535,0.000037,0.000658,0.001067,0.000023,0.071295,0.000102,0.000051,asp_7,0.860609
2,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,B097R46CSY,5.0,1589665266,NaN,Yes!,"Smells good, feels great!",2,True,All Beauty,...,0.002384,0.005362,0.001521,0.000309,0.000172,0.959386,0.000226,0.002328,asp_17,0.959386
3,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,B08BZ63GMJ,5.0,1609322563,NaN,A+,Love it,0,True,All Beauty,...,0.005844,0.034661,0.000687,0.000209,0.000027,0.000094,0.000344,0.002957,asp_9,0.897146
4,AGMJ3EMDVL6OWBJF7CA5RGJLXN5A,B00R8DXL44,4.0,1598567408,NaN,Pretty Color,The polish was quiet thick and did not apply s...,0,True,All Beauty,...,0.014724,0.002269,0.016684,0.269764,0.013187,0.001529,0.002464,0.003324,asp_4,0.480720


In [8]:
data["parent_asin"].value_counts()

parent_asin
B085BB7B1M    1828
B0BM4GX6TT    1609
B07C533XCW    1310
B00R1TAN7I    1309
B019GBG0IE    1197
              ... 
B07PGQ5JH5       1
B085LK54NM       1
B01AO7EXI8       1
B07S1S658F       1
B07ZPVM149       1
Name: count, Length: 95431, dtype: int64

In [9]:
data.columns

Index(['user_id', 'parent_asin', 'rating', 'timestamp', 'history',
       'review_title', 'review_text', 'helpful_vote', 'verified_purchase',
       'main_category', 'item_title', 'average_rating', 'rating_number',
       'features', 'description', 'price', 'store', 'categories', 'details',
       'bought_together', 'asp_0', 'asp_1', 'asp_2', 'asp_3', 'asp_4', 'asp_5',
       'asp_6', 'asp_7', 'asp_8', 'asp_9', 'asp_10', 'asp_11', 'asp_12',
       'asp_13', 'asp_14', 'asp_15', 'asp_16', 'asp_17', 'asp_18', 'asp_19',
       'top_aspect', 'top_aspect_score'],
      dtype='object')

In [10]:
# Suppose phi columns are named like phi_0 ... phi_19
K = 20
phi_cols = [f"asp_{k}" for k in range(K)]
phi_cols

['asp_0',
 'asp_1',
 'asp_2',
 'asp_3',
 'asp_4',
 'asp_5',
 'asp_6',
 'asp_7',
 'asp_8',
 'asp_9',
 'asp_10',
 'asp_11',
 'asp_12',
 'asp_13',
 'asp_14',
 'asp_15',
 'asp_16',
 'asp_17',
 'asp_18',
 'asp_19']

In [11]:
# 1) bootstrap user preference from user's own text (or use history-based)
w_u = estimate_user_pref_from_own_text(data, user_id="AG73BVBKUOH22USSFJA5ZWL7AKXA", phi_cols=phi_cols, beta=8.0)

# 2) extract sentences for a product using gumbel-priority greedy
out = extract_for_user_product(
    df_sent=data,
    user_id="AG73BVBKUOH22USSFJA5ZWL7AKXA",
    product_id="B085BB7B1M",
    phi_cols=phi_cols,
    w_u=w_u,
    k=10,
    L=900,
    lam=0.2,
    tau_ext=2.0,
    seed=42,
)

selected_df = out["selected_df"]
z_t = out["z_t"]
print("Aspect profile z_t:", z_t)

Aspect profile z_t: [0.00579532 0.33999106 0.00783861 0.14473772 0.009862   0.03982417
 0.16980599 0.01092338 0.00483889 0.01586563 0.00451756 0.00069878
 0.00475587 0.01305418 0.01326077 0.06573814 0.05990624 0.00683461
 0.04038483 0.0413662 ]


In [12]:
selected_df[["review_text", "utility", "gumbel_score", "len"]]

,review_text,utility,gumbel_score,len
315349,I forgot how clean a new scrubber felt! These ...,-0.204912,7.045667,262
111784,Have used/loved these for YEARS. Best scrubbi...,-0.007255,7.003428,65
307911,Amazing. This is exactly what I was looking fo...,0.103277,6.783961,113
393774,Excellent!<br /><br />I love these!,-0.006037,6.737782,35
171759,I love these Japanese cleansing cloths. These...,-0.024005,5.974783,130
110070,These are great for exfoliating at home! Nice ...,0.074546,5.968370,77
356524,These work better than any bathing device I ha...,-0.008172,5.688944,106
393689,these are awesome,0.011765,5.282896,17
393594,Helps with my keratosis pillaris. Easy to wash.,0.078931,5.197214,47
393834,Way too big and cumbersome.,0.003883,5.133841,27


In [14]:
# can you help me store out 
selected_df.to_csv("../data/extracted_sentences_gumbel_extractor.csv", index=False)